# VeRI Vehicle Re-Identification — Kaggle → Modeling-Ready CSV

This notebook has been adapted to **VeRI (VeRi-776)** using the Kaggle dataset:

`abhyudaya12/veri-vehicle-re-identification-dataset`

It does **not** use CarDD YOLO damage labels or CompCars make/model folder logic.

**Final output:**  
`/content/drive/MyDrive/capstone-vehicle-metadata/data/processed/VeRI_modeling_ready.csv`

The pipeline:
1. mounts Google Drive (when running in Colab);
2. securely configures Kaggle credentials without hard-coding a token;
3. downloads and extracts VeRI;
4. discovers `image_train`, `image_query`, and `image_test` even if Kaggle adds extra nesting;
5. parses `vehicle_id` and `camera_id` from VeRI filenames;
6. optionally joins `train_label.xml` / `test_label.xml` attributes (`colorID`, `typeID`) when available;
7. checks image dimensions/corruption;
8. writes one modeling-ready CSV to `data/processed`.


In [ ]:
# Setup paths and imports
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys
import zipfile
import hashlib
import xml.etree.ElementTree as ET

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

# Mount Drive when running in Google Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/capstone-vehicle-metadata')
    RUNNING_IN_COLAB = True
except Exception:
    # Local/Jupyter fallback.
    PROJECT_ROOT = Path.cwd() / 'capstone-vehicle-metadata'
    RUNNING_IN_COLAB = False

DATASET_DIR = PROJECT_ROOT / 'data' / 'veri_kaggle'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PERSISTENT_EXTRACT_DIR = DATASET_DIR / 'extracted'

LOCAL_EXTRACT_DIR = Path('/content/veri_local') if Path('/content').exists() else (DATASET_DIR / 'local_extract')
KAGGLE_DATASET = 'abhyudaya12/veri-vehicle-re-identification-dataset'
OUTPUT_CSV = PROCESSED_DIR / 'VeRI_modeling_ready.csv'

DATASET_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATASET_DIR:', DATASET_DIR)
print('OUTPUT_CSV:', OUTPUT_CSV)


## 1. Kaggle access

In [ ]:
# Install Kaggle CLI and configure credentials securely.
# No API token is stored in this notebook.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)

has_access_token = (kaggle_dir / 'access_token').exists()
has_kaggle_json = (kaggle_dir / 'kaggle.json').exists()

if not (has_access_token or has_kaggle_json):
    token = None

    # First try a Colab secret called KAGGLE_API_TOKEN.
    if RUNNING_IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('KAGGLE_API_TOKEN')
        except Exception:
            token = None

    # Otherwise ask securely at runtime (input is hidden).
    if not token:
        from getpass import getpass
        token = getpass('Paste your Kaggle API token (hidden): ').strip()

    if not token:
        raise RuntimeError(
            'No Kaggle credentials found. Add KAGGLE_API_TOKEN to Colab Secrets, '
            'or paste your token when prompted.'
        )

    access_token_path = kaggle_dir / 'access_token'
    access_token_path.write_text(token)
    os.chmod(access_token_path, 0o600)

print('Kaggle credentials are configured.')


## 2. Download and extract VeRI

In [ ]:
# Download the VeRI Kaggle dataset.
zip_candidates_before = set(DATASET_DIR.glob('*.zip'))

cmd = [
    'kaggle', 'datasets', 'download',
    '-d', KAGGLE_DATASET,
    '-p', str(DATASET_DIR),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

zip_files = sorted(DATASET_DIR.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if not zip_files:
    raise FileNotFoundError(f'No ZIP file found in {DATASET_DIR} after Kaggle download.')

ZIP_PATH = zip_files[0]
print('Downloaded ZIP:', ZIP_PATH)
print(f'ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.1f} MB')


In [ ]:
# Extract to local disk for faster scanning.
# Re-running is safe: existing extracted files are reused.
LOCAL_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    members = zf.infolist()
    print(f'Archive entries: {len(members):,}')
    for member in tqdm(members, desc='Extracting VeRI'):
        target = LOCAL_EXTRACT_DIR / member.filename
        if target.exists() and target.is_file() and target.stat().st_size == member.file_size:
            continue
        zf.extract(member, path=LOCAL_EXTRACT_DIR)

print('Extraction complete:', LOCAL_EXTRACT_DIR)


## 3. Detect VeRI structure

In [ ]:
# Discover VeRI folders regardless of extra Kaggle nesting.
def find_named_directory(search_root: Path, folder_name: str):
    matches = [p for p in search_root.rglob(folder_name) if p.is_dir()]
    if not matches:
        return None
    # Prefer the directory containing the most image files.
    image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
    def image_count(p):
        return sum(1 for x in p.iterdir() if x.is_file() and x.suffix.lower() in image_exts)
    return max(matches, key=image_count)

split_dirs = {
    'train': find_named_directory(LOCAL_EXTRACT_DIR, 'image_train'),
    'query': find_named_directory(LOCAL_EXTRACT_DIR, 'image_query'),
    'test': find_named_directory(LOCAL_EXTRACT_DIR, 'image_test'),
}

for split, path in split_dirs.items():
    print(f'{split:>5}: {path}')

missing = [s for s, p in split_dirs.items() if p is None]
if missing:
    raise FileNotFoundError(
        f'Missing expected VeRI folder(s): {missing}. '
        f'Inspect {LOCAL_EXTRACT_DIR} to confirm the Kaggle archive structure.'
    )

# Infer dataset root from the split folders.
candidate_roots = [p.parent for p in split_dirs.values()]
VE_RI_ROOT = max(set(candidate_roots), key=candidate_roots.count)
print('Detected VeRI root:', VE_RI_ROOT)


In [ ]:
# Helpers for VeRI filename parsing and optional XML attributes.
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# Standard VeRI example: 0001_c001_00016450_0.jpg
VERI_NAME_RE = re.compile(
    r'^(?P<vehicle_id>-?\d+)_c(?P<camera_id>\d+)(?:_(?P<tail>.*))?$',
    re.IGNORECASE
)

def parse_veri_filename(filename: str):
    stem = Path(filename).stem
    m = VERI_NAME_RE.match(stem)
    if not m:
        return {'vehicle_id': pd.NA, 'camera_id': pd.NA, 'filename_tail': pd.NA}
    return {
        'vehicle_id': int(m.group('vehicle_id')),
        'camera_id': int(m.group('camera_id')),
        'filename_tail': m.group('tail') if m.group('tail') else pd.NA,
    }

def locate_file(root: Path, filename: str):
    matches = list(root.rglob(filename))
    return matches[0] if matches else None

def read_id_name_list(path: Path | None):
    """Read files such as list_color.txt or list_type.txt when present."""
    mapping = {}
    if path is None or not path.exists():
        return mapping
    for raw in path.read_text(errors='ignore').splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = re.split(r'[\s,]+', line, maxsplit=1)
        if len(parts) == 2 and parts[0].lstrip('-').isdigit():
            mapping[int(parts[0])] = parts[1].strip()
    return mapping

FALLBACK_COLOR = {
    1: 'yellow', 2: 'orange', 3: 'green', 4: 'gray', 5: 'red',
    6: 'blue', 7: 'white', 8: 'golden', 9: 'brown', 10: 'black'
}
FALLBACK_TYPE = {
    1: 'sedan', 2: 'suv', 3: 'van', 4: 'hatchback', 5: 'mpv',
    6: 'pickup', 7: 'bus', 8: 'truck', 9: 'estate'
}

color_file = locate_file(VE_RI_ROOT, 'list_color.txt')
type_file = locate_file(VE_RI_ROOT, 'list_type.txt')
color_map = read_id_name_list(color_file) or FALLBACK_COLOR
type_map = read_id_name_list(type_file) or FALLBACK_TYPE

print('Color labels:', color_map)
print('Type labels:', type_map)


## 4. Read optional VeRI attribute labels

In [ ]:
# Parse train/test XML labels if the Kaggle copy includes them.
def parse_label_xml(xml_path: Path | None, annotation_split: str):
    rows = []
    if xml_path is None or not xml_path.exists():
        return pd.DataFrame()

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for item in root.iter():
        attrs = item.attrib
        if 'imageName' not in attrs:
            continue

        def to_int_or_na(value):
            if value is None or value == '':
                return pd.NA
            digits = re.sub(r'^[cC]', '', str(value))
            return int(digits) if digits.lstrip('-').isdigit() else pd.NA

        rows.append({
            'filename': attrs.get('imageName'),
            'xml_vehicle_id': to_int_or_na(attrs.get('vehicleID')),
            'xml_camera_id': to_int_or_na(attrs.get('cameraID')),
            'color_id': to_int_or_na(attrs.get('colorID')),
            'type_id': to_int_or_na(attrs.get('typeID')),
            'annotation_split': annotation_split,
        })

    return pd.DataFrame(rows)

train_xml = locate_file(VE_RI_ROOT, 'train_label.xml')
test_xml = locate_file(VE_RI_ROOT, 'test_label.xml')

xml_frames = [
    parse_label_xml(train_xml, 'train'),
    parse_label_xml(test_xml, 'test'),
]
xml_frames = [x for x in xml_frames if not x.empty]
df_xml = pd.concat(xml_frames, ignore_index=True) if xml_frames else pd.DataFrame()

if not df_xml.empty:
    # Filename is normally unique for the annotation lookup.
    df_xml = df_xml.drop_duplicates(subset='filename', keep='first')
    df_xml['color_name'] = df_xml['color_id'].map(color_map)
    df_xml['type_name'] = df_xml['type_id'].map(type_map)
    print(f'Parsed XML labels for {len(df_xml):,} images.')
    display(df_xml.head())
else:
    print('No train_label.xml/test_label.xml found. Filename IDs will still be used.')


## 5. Build image inventory

In [ ]:
# Build VeRI inventory from train/query/test images.
records = []

for split, img_dir in split_dirs.items():
    image_files = sorted(
        p for p in img_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

    print(f'{split}: {len(image_files):,} images')

    for img_path in tqdm(image_files, desc=f'Processing {split}'):
        ids = parse_veri_filename(img_path.name)

        try:
            with Image.open(img_path) as img:
                width, height = img.size
            corrupted = False
        except Exception:
            width, height = pd.NA, pd.NA
            corrupted = True

        records.append({
            'local_file_path': str(img_path),
            'relative_path': str(img_path.relative_to(VE_RI_ROOT)),
            'filename': img_path.name,
            'split': split,
            'vehicle_id': ids['vehicle_id'],
            'camera_id': ids['camera_id'],
            'filename_tail': ids['filename_tail'],
            'width': width,
            'height': height,
            'size_kb': round(img_path.stat().st_size / 1024, 2),
            'corrupted': corrupted,
            'source_dataset': 'VeRI',
        })

df_veri = pd.DataFrame(records)

if df_veri.empty:
    raise RuntimeError('No VeRI images were found.')

print(f'Total images indexed: {len(df_veri):,}')
print(df_veri['split'].value_counts())
print(f'Unparsed vehicle IDs: {df_veri["vehicle_id"].isna().sum():,}')
print(f'Unparsed camera IDs: {df_veri["camera_id"].isna().sum():,}')
print(f'Corrupted images: {df_veri["corrupted"].sum():,}')
display(df_veri.head())


In [ ]:
# Join XML attributes when available.
if not df_xml.empty:
    join_cols = [
        'filename', 'xml_vehicle_id', 'xml_camera_id',
        'color_id', 'type_id', 'color_name', 'type_name'
    ]
    df_veri = df_veri.merge(df_xml[join_cols], on='filename', how='left')

    # Keep filename-derived IDs as the canonical values; use XML IDs only to check consistency.
    vehicle_mismatch = (
        df_veri['xml_vehicle_id'].notna()
        & df_veri['vehicle_id'].notna()
        & (df_veri['xml_vehicle_id'].astype('Int64') != df_veri['vehicle_id'].astype('Int64'))
    )
    camera_mismatch = (
        df_veri['xml_camera_id'].notna()
        & df_veri['camera_id'].notna()
        & (df_veri['xml_camera_id'].astype('Int64') != df_veri['camera_id'].astype('Int64'))
    )
    print('Vehicle-ID filename/XML mismatches:', int(vehicle_mismatch.sum()))
    print('Camera-ID filename/XML mismatches:', int(camera_mismatch.sum()))
else:
    for c in ['color_id', 'type_id', 'color_name', 'type_name']:
        df_veri[c] = pd.NA

# Use nullable integer dtype so missing values are represented cleanly.
for c in ['vehicle_id', 'camera_id', 'color_id', 'type_id']:
    df_veri[c] = pd.to_numeric(df_veri[c], errors='coerce').astype('Int64')

display(df_veri.head())


## 6. Quality checks

In [ ]:
# Modeling-ready quality checks.
print('Rows:', f'{len(df_veri):,}')
print('Unique vehicle IDs:', df_veri['vehicle_id'].nunique(dropna=True))
print('Unique camera IDs:', df_veri['camera_id'].nunique(dropna=True))
print('Corrupted:', int(df_veri['corrupted'].sum()))
print('Duplicate file paths:', int(df_veri['local_file_path'].duplicated().sum()))

print('\nImages by split:')
print(df_veri['split'].value_counts())

print('\nTop vehicle IDs by image count:')
print(df_veri['vehicle_id'].value_counts().head(10))

if df_veri['color_name'].notna().any():
    print('\nColor distribution:')
    print(df_veri['color_name'].value_counts(dropna=False))

if df_veri['type_name'].notna().any():
    print('\nVehicle type distribution:')
    print(df_veri['type_name'].value_counts(dropna=False))


## 7. Save persistent dataset paths + modeling-ready CSV

In [ ]:
# Persist the extracted VeRI data to Drive so CSV file paths remain valid after Colab ends.
# This can take time the first time because VeRI contains many image files.
PERSISTENT_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print('Copying/reusing extracted dataset in Drive...')
shutil.copytree(LOCAL_EXTRACT_DIR, PERSISTENT_EXTRACT_DIR, dirs_exist_ok=True)
print('Persistent dataset location:', PERSISTENT_EXTRACT_DIR)


In [ ]:
# Convert local paths to persistent Drive paths and save the final modeling-ready CSV.
def local_to_persistent(local_path: str):
    p = Path(local_path)
    rel = p.relative_to(LOCAL_EXTRACT_DIR)
    return str(PERSISTENT_EXTRACT_DIR / rel)

df_modeling = df_veri.copy()
df_modeling['file_path'] = df_modeling['local_file_path'].map(local_to_persistent)

# Keep only rows whose image opened successfully. Do NOT deduplicate across query/test:
# VeRI's re-identification protocol depends on preserving the official split.
df_modeling = df_modeling[~df_modeling['corrupted']].copy()

preferred_cols = [
    'file_path', 'relative_path', 'filename', 'split',
    'vehicle_id', 'camera_id', 'filename_tail',
    'color_id', 'color_name', 'type_id', 'type_name',
    'width', 'height', 'size_kb', 'corrupted', 'source_dataset'
]
df_modeling = df_modeling[[c for c in preferred_cols if c in df_modeling.columns]]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_modeling.to_csv(OUTPUT_CSV, index=False)

print(f'Saved {len(df_modeling):,} rows to:')
print(OUTPUT_CSV)
display(df_modeling.head())


In [ ]:
# Verify the saved CSV.
df_check = pd.read_csv(OUTPUT_CSV)

print('Saved rows:', f'{len(df_check):,}')
print('Columns:', df_check.columns.tolist())
print('\nSplit counts:')
print(df_check['split'].value_counts())
print('\nSource dataset values:', df_check['source_dataset'].unique().tolist())

assert len(df_check) > 0, 'The saved CSV is empty.'
assert set(df_check['source_dataset'].dropna().unique()) == {'VeRI'}
assert {'train', 'query', 'test'}.issubset(set(df_check['split'].unique()))
assert df_check['vehicle_id'].notna().any()
assert df_check['camera_id'].notna().any()

print('\nVerification passed.')
display(df_check.head(3))


## Notes for modeling

- `vehicle_id` is the VeRI identity label.
- `camera_id` is the camera identifier parsed from filenames such as `0001_c001_...jpg`.
- `split` preserves VeRI's official `train`, `query`, and `test` layout.
- `color_name` and `type_name` are populated when `train_label.xml` / `test_label.xml` are available in the Kaggle copy.
- Query/test rows are intentionally **not deduplicated**, because that could break the official re-identification evaluation protocol.
- The generated CSV is `data/processed/VeRI_modeling_ready.csv`.
